# Emergency Text XLM-R Full Fine-Tuning

Runs a sealed, family-disjoint multilingual road-accident text experiment. The included dataset is synthetic/draft, so every result remains provisional and the active model is never changed.


## SAVE RESULTS TO GOOGLE DRIVE — run the next cell only

This cell reconnects Drive, displays the saved results, and saves **My Drive → emergency_text_xlmr_summary.zip**. It works independently of the other cells. It does not retrain or change the existing reports. If a cell is still running, stop it first; then run only the cell below. Do not use **Run All**.


In [ ]:
# SAVE RESULTS TO GOOGLE DRIVE - run only this cell
# Reads the existing training outputs; does not train or rewrite reports.
from pathlib import Path
import hashlib
import json
import shutil
import tempfile
import zipfile
from google.colab import drive

MY_DRIVE = Path("/content/drive/MyDrive")
if not MY_DRIVE.is_dir():
    print("Connecting Google Drive...")
    drive.mount("/content/drive")

PACKAGE_ID = "emergency-text-xlmr-provisional-20261004"
RUN_ROOT = MY_DRIVE / "emergency_text_xlmr_runs" / PACKAGE_ID

if not RUN_ROOT.is_dir():
    raise FileNotFoundError(
        f"Saved training folder not found: {RUN_ROOT}. "
        "Connect the Google account used for training."
    )

for relative in (
    "BEST_CHECKPOINT.json",
    "best_model_test_evaluation/evaluation_metrics.json",
):
    if not (RUN_ROOT / relative).is_file():
        raise FileNotFoundError(
            f"Missing saved result: {relative}. Stop here; do not retrain."
        )

best_info = json.loads((RUN_ROOT / "BEST_CHECKPOINT.json").read_text(encoding="utf-8"))
print("Selected seed:", best_info["seed"])

# Show original reports, without recreating completion or approval records.
for filename in ("ACCEPTANCE_REPORT.json", "RUN_COMPLETE.json"):
    path = RUN_ROOT / filename
    print(f"\n{filename} EXISTS: {path.is_file()}")
    if path.is_file():
        print(json.dumps(json.loads(path.read_text(encoding="utf-8")), indent=2))

# Package JSON reports only. Exclude the large tokenizer vocabulary.
report_paths = sorted(
    path for path in RUN_ROOT.rglob("*.json")
    if path.is_file() and path.name != "tokenizer.json"
)
print(f"Creating summary from {len(report_paths)} JSON files...")
summary_zip = Path("/content/emergency_text_xlmr_summary.zip")

with tempfile.TemporaryDirectory(prefix="emergency_text_export_", dir="/content") as temporary:
    temporary_zip = Path(temporary) / summary_zip.name
    with zipfile.ZipFile(temporary_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in report_paths:
            content = path.read_bytes()
            json.loads(content)  # Stop if a saved report is incomplete/corrupt.
            archive.writestr(path.relative_to(RUN_ROOT).as_posix(), content)
    with zipfile.ZipFile(temporary_zip) as archive:
        if archive.testzip() is not None:
            raise RuntimeError("Summary ZIP validation failed.")
    temporary_zip.replace(summary_zip)

# Save to My Drive; no browser download widget is needed in VS Code.
drive_zip = MY_DRIVE / summary_zip.name
pending_zip = MY_DRIVE / (summary_zip.name + ".partial")
try:
    shutil.copy2(summary_zip, pending_zip)
    if hashlib.sha256(summary_zip.read_bytes()).digest() != hashlib.sha256(pending_zip.read_bytes()).digest():
        raise OSError("Drive copy verification failed.")
    pending_zip.replace(drive_zip)
except OSError as error:
    print(f"Could not save to Google Drive: {error}")
    print(f"The verified ZIP is still available in this runtime: {summary_zip}")
    raise RuntimeError(
        "Drive export did not finish. If Drive is full, free space and rerun only this cell."
    ) from error

print("\nSAVED TO GOOGLE DRIVE")
print("My Drive > emergency_text_xlmr_summary.zip")
print(f"ZIP size: {drive_zip.stat().st_size:,} bytes")


In [4]:
from google.colab import drive
from pathlib import Path
import json

drive.mount("/content/drive")

run_root = Path(
    "/content/drive/MyDrive/emergency_text_xlmr_runs/"
    "emergency-text-xlmr-provisional-20261004"
)

required_files = [
    "BEST_CHECKPOINT.json",
    "xlmr_seed_42/metadata.json",
    "xlmr_seed_42/validation_metrics.json",
    "best_model_test_evaluation/evaluation_metrics.json",
    "ACCEPTANCE_REPORT.json",
    "RUN_COMPLETE.json",
]

missing = []

for relative in required_files:
    path = run_root / relative
    exists = path.is_file()
    print(("✅" if exists else "❌"), relative)

    if not exists:
        missing.append(relative)

complete_flag = False
run_complete_path = run_root / "RUN_COMPLETE.json"

if run_complete_path.is_file():
    run_info = json.loads(run_complete_path.read_text(encoding="utf-8"))
    complete_flag = run_info.get("complete") is True
    print("\nRUN_COMPLETE complete:", complete_flag)

if not missing and complete_flag:
    print("\n✅ TRAINING AND TEST EVALUATION COMPLETE")
    print("Best model: xlmr_seed_42")
else:
    print("\n❌ INCOMPLETE")
    print("Missing files:", missing)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ BEST_CHECKPOINT.json
✅ xlmr_seed_42/metadata.json
✅ xlmr_seed_42/validation_metrics.json
✅ best_model_test_evaluation/evaluation_metrics.json
✅ ACCEPTANCE_REPORT.json
✅ RUN_COMPLETE.json

RUN_COMPLETE complete: True

✅ TRAINING AND TEST EVALUATION COMPLETE
Best model: xlmr_seed_42


In [5]:
for filename in ["ACCEPTANCE_REPORT.json", "RUN_COMPLETE.json"]:
    path = RUN_ROOT / filename
    print(filename, "EXISTS:", path.is_file())
    if path.is_file():
        print(path.read_text(encoding="utf-8"))

NameError: name 'RUN_ROOT' is not defined

In [6]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

RUN_ROOT = Path(
    "/content/drive/MyDrive/emergency_text_xlmr_runs/"
    "emergency-text-xlmr-provisional-20261004"
)

print("Training folder found:", RUN_ROOT.is_dir())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Training folder found: True


In [7]:
from google.colab import files
files.download("/content/emergency_text_xlmr_summary.zip")

FileNotFoundError: Cannot find file: /content/emergency_text_xlmr_summary.zip

In [8]:
# Fixed, audited experiment settings. Do not change labels or split after training starts.
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, shutil, subprocess, sys, zipfile

PACKAGE_ID = "emergency-text-xlmr-provisional-20261004"
BUNDLE_NAME = "emergency_text_xlmr_provisional_20261004.zip"
EXPECTED_BUNDLE_SHA256 = "197f16b0cf0a01867c3dae2f5a5945724214b254d6f7d054da8b5c2b7854969d"
EXPECTED_MANIFEST_SHA256 = "a2fd2f398299669de786fe8015adc3151efc4bedc0f2fdda1bee9e2be306241a"
BASE_MODEL = "FacebookAI/xlm-roberta-base"
BASE_MODEL_REVISION = "42f548f32366559214515ec137cdd16002968bf6"
SEEDS = [41, 42, 43]
EPOCHS = 8
EARLY_STOPPING_PATIENCE = 2
BATCH_SIZE = 8
GRADIENT_ACCUMULATION = 2
MAX_LENGTH = 128
LEARNING_RATE = 2e-5
REVIEW_THRESHOLD = 0.65
USE_GOOGLE_DRIVE = True
RUN_ID = PACKAGE_ID

def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


## 1. Upload and verify the sealed bundle


In [9]:
from google.colab import files

uploaded = files.upload()
if BUNDLE_NAME not in uploaded:
    raise RuntimeError(f"Upload the exact bundle named {BUNDLE_NAME}")
bundle_path = Path("/content") / BUNDLE_NAME
bundle_path.write_bytes(uploaded[BUNDLE_NAME])
actual_bundle_sha256 = file_sha256(bundle_path)
if actual_bundle_sha256 != EXPECTED_BUNDLE_SHA256:
    raise RuntimeError("Bundle SHA-256 mismatch; stop and rebuild the package")

BUNDLE_DIR = Path("/content/emergency_text_xlmr_bundle")
if BUNDLE_DIR.exists():
    shutil.rmtree(BUNDLE_DIR)
BUNDLE_DIR.mkdir(parents=True)
with zipfile.ZipFile(bundle_path) as archive:
    root = BUNDLE_DIR.resolve()
    for member in archive.infolist():
        destination = (BUNDLE_DIR / member.filename).resolve()
        if root not in destination.parents and destination != root:
            raise RuntimeError("Unsafe path in ZIP package")
    archive.extractall(BUNDLE_DIR)

manifest_path = BUNDLE_DIR / "package_manifest.json"
if file_sha256(manifest_path) != EXPECTED_MANIFEST_SHA256:
    raise RuntimeError("Package manifest checksum mismatch")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
for relative, expected in manifest["files_sha256"].items():
    path = BUNDLE_DIR / relative
    if not path.is_file() or file_sha256(path) != expected:
        raise RuntimeError(f"Missing or changed package file: {relative}")
print(json.dumps({"package": PACKAGE_ID, "bundle_verified": True, "dataset": manifest["dataset"]}, indent=2))


RuntimeError: Upload the exact bundle named emergency_text_xlmr_provisional_20261004.zip

In [ ]:
from google.colab import files
files.download("/content/emergency_text_xlmr_summary.zip")

## 2. Install pinned-compatible dependencies and require a GPU


In [ ]:
subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
    "-r", str(BUNDLE_DIR / "requirements-text-colab.txt")
], check=True)

import torch, transformers, sklearn, safetensors
if not torch.cuda.is_available():
    raise RuntimeError("GPU is required. In Colab choose Runtime > Change runtime type > GPU, then run again.")
print({
    "gpu": torch.cuda.get_device_name(0),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "scikit_learn": sklearn.__version__,
    "safetensors": safetensors.__version__,
})


## 3. Mount Drive and create or resume this exact run


In [ ]:
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_BASE = Path("/content/drive/MyDrive/emergency_text_xlmr_runs")
else:
    OUTPUT_BASE = Path("/content/emergency_text_xlmr_runs")

RUN_ROOT = OUTPUT_BASE / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=True)
receipt_path = RUN_ROOT / "RUN_RECEIPT.json"
receipt = {
    "package_id": PACKAGE_ID,
    "bundle_sha256": EXPECTED_BUNDLE_SHA256,
    "mode": manifest["mode"],
    "started_at": datetime.now(timezone.utc).isoformat(),
    "active_model_changed": False,
    "approved_for_deployment": False,
}
if receipt_path.exists():
    existing = json.loads(receipt_path.read_text(encoding="utf-8"))
    if existing.get("bundle_sha256") != EXPECTED_BUNDLE_SHA256:
        raise RuntimeError("RUN_ID already belongs to a different package")
else:
    receipt_path.write_text(json.dumps(receipt, indent=2) + "\n", encoding="utf-8")
print("Run output:", RUN_ROOT)


## 4. Validate dataset and frozen split before any training


In [ ]:
DATA = BUNDLE_DIR / "data/reports.csv"
SPLIT = BUNDLE_DIR / "data/frozen_split_manifest.json"
TRAIN = BUNDLE_DIR / "train_text.py"

subprocess.run([sys.executable, str(TRAIN), "validate", "--data", str(DATA)], cwd=BUNDLE_DIR, check=True)

split = json.loads(SPLIT.read_text(encoding="utf-8"))
group_sets = [set(split[name]["group_hashes"]) for name in ("train", "validation", "test")]
text_sets = [set(split[name]["text_hashes"]) for name in ("train", "validation", "test")]
if any(group_sets[i] & group_sets[j] for i in range(3) for j in range(i + 1, 3)):
    raise RuntimeError("Frozen split has group leakage")
if any(text_sets[i] & text_sets[j] for i in range(3) for j in range(i + 1, 3)):
    raise RuntimeError("Frozen split has normalized-text leakage")
print({name: split[name]["summary"] for name in ("train", "validation", "test")})


In [ ]:
def run_checked(command):
    print("RUN:", " ".join(map(str, command)))
    subprocess.run([str(item) for item in command], cwd=BUNDLE_DIR, check=True)

def completed_model(path):
    required = ["metadata.json", "split_manifest.json", "validation_metrics.json"]
    return all((path / name).is_file() for name in required)

def run_then_publish(command, final_output):
    if completed_model(final_output):
        print("Already complete; skipping:", final_output)
        return
    if final_output.exists() and any(final_output.iterdir()):
        raise RuntimeError(f"Incomplete output exists and was not overwritten: {final_output}")
    local_output = Path("/content/current_training_output")
    if local_output.exists():
        shutil.rmtree(local_output)
    command = [str(local_output) if str(item) == "__OUTPUT__" else item for item in command]
    run_checked(command)
    if not completed_model(local_output):
        raise RuntimeError("Training command finished without a complete artifact")
    shutil.copytree(local_output, final_output)
    shutil.rmtree(local_output)


## 5. Train the required TF-IDF baseline


In [ ]:
baseline_output = RUN_ROOT / "baseline_seed_42"
run_then_publish([
    sys.executable, TRAIN, "baseline", "--data", DATA, "--output", "__OUTPUT__",
    "--allow-draft", "--seed", "42", "--split-manifest", SPLIT,
    "--review-threshold", str(REVIEW_THRESHOLD)
], baseline_output)
print(json.loads((baseline_output / "validation_metrics.json").read_text(encoding="utf-8"))["raw_classifier"])


## 6. Full fine-tune XLM-R with three seeds

Completed seed directories are reused on rerun. An interrupted in-memory seed is repeated without overwriting a completed artifact.


In [ ]:
for seed in SEEDS:
    final_output = RUN_ROOT / f"xlmr_seed_{seed}"
    run_then_publish([
        sys.executable, TRAIN, "train", "--data", DATA, "--output", "__OUTPUT__",
        "--allow-draft", "--seed", str(seed), "--split-manifest", SPLIT,
        "--base-model", BASE_MODEL, "--revision", BASE_MODEL_REVISION, "--epochs", str(EPOCHS),
        "--early-stopping-patience", str(EARLY_STOPPING_PATIENCE),
        "--selection-metric", "critical_recall_then_macro_f1",
        "--batch-size", str(BATCH_SIZE), "--gradient-accumulation", str(GRADIENT_ACCUMULATION),
        "--max-length", str(MAX_LENGTH), "--unfreeze-backbone",
        "--learning-rate", str(LEARNING_RATE), "--threads", "2", "--device", "cuda",
        "--review-threshold", str(REVIEW_THRESHOLD)
    ], final_output)


## 7. Select on validation, then evaluate the untouched test split once


In [ ]:
candidates = []
for seed in SEEDS:
    path = RUN_ROOT / f"xlmr_seed_{seed}"
    metrics = json.loads((path / "validation_metrics.json").read_text(encoding="utf-8"))
    raw = metrics["raw_classifier"]
    score = (raw["per_class"]["critical"]["recall"], raw["macro_f1"])
    candidates.append((score, seed, path))
candidates.sort(reverse=True, key=lambda item: (item[0], -item[1]))
best_score, best_seed, best_model = candidates[0]
(RUN_ROOT / "BEST_CHECKPOINT.json").write_text(json.dumps({
    "seed": best_seed, "path": str(best_model),
    "validation_critical_recall": best_score[0], "validation_macro_f1": best_score[1],
    "selection_rule": "critical_recall_then_macro_f1",
}, indent=2) + "\n", encoding="utf-8")

test_output = RUN_ROOT / "best_model_test_evaluation"
if not (test_output / "evaluation_metrics.json").is_file():
    if test_output.exists() and any(test_output.iterdir()):
        raise RuntimeError("Incomplete test evaluation exists and was not overwritten")
    run_checked([
        sys.executable, TRAIN, "evaluate", "--data", DATA, "--model", best_model,
        "--output", test_output, "--split", "test", "--allow-draft",
        "--batch-size", "16", "--device", "cuda"
    ])
print("Best checkpoint:", best_model)


In [ ]:
best_info = json.loads(
    (RUN_ROOT / "BEST_CHECKPOINT.json").read_text(encoding="utf-8")
)
print(json.dumps(best_info, indent=2))


## 8. Apply acceptance gates without activating or deploying the model


In [ ]:
metrics = json.loads((test_output / "evaluation_metrics.json").read_text(encoding="utf-8"))
raw = metrics["raw_classifier"]
confusion = raw["confusion_matrix"]["rows_actual_columns_predicted"]
labels = raw["confusion_matrix"]["labels"]
serious_i, critical_i = labels.index("serious"), labels.index("critical")
urgent_support = sum(confusion[serious_i]) + sum(confusion[critical_i])
urgent_detected = sum(confusion[serious_i][i] + confusion[critical_i][i] for i in (serious_i, critical_i))
urgent_recall = urgent_detected / urgent_support if urgent_support else 0.0
prediction_rows = json.loads((test_output / "evaluation_predictions.json").read_text(encoding="utf-8"))["rows"]
def raw_macro_f1(selected_rows):
    actual = [row["label"] for row in selected_rows]
    predicted = [labels[max(range(len(row["probabilities"])), key=lambda i: row["probabilities"][i])] for row in selected_rows]
    if not actual:
        return None
    scores = []
    for label in labels:
        true_positive = sum(a == label and p == label for a, p in zip(actual, predicted))
        support = sum(a == label for a in actual)
        predicted_count = sum(p == label for p in predicted)
        precision = true_positive / predicted_count if predicted_count else 0.0
        recall = true_positive / support if support else 0.0
        scores.append(2 * precision * recall / (precision + recall) if precision + recall else 0.0)
    return sum(scores) / len(labels)

language_f1 = {
    language: raw_macro_f1([row for row in prediction_rows if row["language"] == language])
    for language in sorted({row["language"] for row in prediction_rows})
}
metric_checks = {
    "overall_macro_f1_at_least_0_80": raw["macro_f1"] >= 0.80,
    "critical_recall_at_least_0_95": raw["per_class"]["critical"]["recall"] >= 0.95,
    "serious_or_critical_detection_recall_at_least_0_90": urgent_recall >= 0.90,
    "every_observed_language_macro_f1_at_least_0_75": all(value >= 0.75 for value in language_f1.values()),
    "raw_high_risk_undertriage_is_zero": metrics["raw_high_risk_undertriage_count"] == 0,
}
all_data_reviewed = bool(metrics.get("all_data_reviewed"))
report = {
    "package_id": PACKAGE_ID,
    "mode": manifest["mode"],
    "best_seed": best_seed,
    "best_checkpoint": str(best_model),
    "test_metrics": {
        "macro_f1": raw["macro_f1"],
        "critical_recall": raw["per_class"]["critical"]["recall"],
        "serious_or_critical_detection_recall": urgent_recall,
        "per_language_macro_f1": language_f1,
        "raw_high_risk_undertriage_count": metrics["raw_high_risk_undertriage_count"],
    },
    "metric_checks": metric_checks,
    "all_data_reviewed": all_data_reviewed,
    "metrics_passed": all(metric_checks.values()),
    "approved_for_deployment": False,
    "active_model_changed": False,
    "status": "provisional_research_complete_not_deployable" if not all_data_reviewed else "reviewed_experiment_requires_human_promotion_decision",
    "completed_at": datetime.now(timezone.utc).isoformat(),
}
(RUN_ROOT / "ACCEPTANCE_REPORT.json").write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
(RUN_ROOT / "RUN_COMPLETE.json").write_text(json.dumps({
    "complete": True, "package_id": PACKAGE_ID, "best_seed": best_seed,
    "approved_for_deployment": False, "active_model_changed": False,
    "completed_at": datetime.now(timezone.utc).isoformat(),
}, indent=2) + "\n", encoding="utf-8")
print(json.dumps(report, indent=2))


## 9. Create a small metrics-only download


In [ ]:
summary_root = Path("/content/emergency_text_xlmr_summary")
if summary_root.exists():
    shutil.rmtree(summary_root)
summary_root.mkdir()
for path in RUN_ROOT.rglob("*.json"):
    relative = path.relative_to(RUN_ROOT)
    destination = summary_root / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(path, destination)
summary_zip = shutil.make_archive("/content/emergency_text_xlmr_summary", "zip", summary_root)
print("Metrics summary ZIP:", summary_zip)
print("Full best model remains in Google Drive:", best_model)
